# Interactive Solar Forecasting — XGBoost vs Actual (Plotly)

Compare **XGBoost predictions** against real inverter data using an interactive Plotly chart.

- **System:** 3.57 kW rooftop, Karnal, Haryana
- **Window:** last 14 days of the dataset where `power_now_w > 0`
- **Output:** `solar_forecast.html` — a fully interactive standalone chart

Requires `solar_xgb_model.json` produced by `02_xgboost_model.ipynb`.

In [ ]:
!pip install xgboost shap plotly influxdb-client --quiet

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
# ── Load data and model ────────────────────────────────────────────────────────
%run data_loader.py
df = load_solar_data()
df['datetime'] = pd.to_datetime(df['datetime'])
df['date']     = pd.to_datetime(df['date'])

FEATURES = ['hour', 'month', 'day_of_year',
            'irradiance', 'ambient_temp', 'module_temp']
TARGET   = 'power_now_w'

model = XGBRegressor()
model.load_model('solar_xgb_model.json')

print(f'Loaded {len(df):,} rows. Model ready.')

In [ ]:
# ── Select the last 14 days where power_now_w > 0 ─────────────────────────────
df_active = df[df['power_now_w'] > 0].dropna(subset=FEATURES + [TARGET]).copy()
df_active = df_active.sort_values('datetime').reset_index(drop=True)

# Identify the last calendar date with readings, step back 14 days
last_date  = df_active['date'].max()
start_date = last_date - pd.Timedelta(days=13)  # 14 days inclusive

window = df_active[df_active['date'] >= start_date].copy()

print(f'Forecast window: {start_date.date()} → {last_date.date()}')
print(f'Rows in window : {len(window):,}')

In [ ]:
# ── Build predictions for the 14-day window ────────────────────────────────────
X_window = window[FEATURES].values
window   = window.copy()
window['predicted_w'] = model.predict(X_window)
window['residual_w']  = window[TARGET] - window['predicted_w']

print(f'Prediction done. Sample:')
window[['datetime', TARGET, 'predicted_w', 'residual_w']].tail(5)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  BUILD PLOTLY FIGURE  (2 rows, shared x-axis)
# ══════════════════════════════════════════════════════════════════════════════
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.06,
    subplot_titles=[
        'Solar Generation: Actual vs XGBoost Prediction',
        'Prediction Error (W)'
    ]
)

# ─────────────────────────────────────────────────────────────────────────────
# ROW 1 — Actual power (blue solid) + Predicted power (orange dashed)
# ─────────────────────────────────────────────────────────────────────────────
fig.add_trace(
    go.Scatter(
        x=window['datetime'],
        y=window[TARGET],
        mode='lines',
        name='Actual (W)',
        line=dict(color='#1565C0', width=1.5),
        hovertemplate='%{x|%Y-%m-%d %H:%M}<br>Actual: %{y:.0f} W<extra></extra>'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=window['datetime'],
        y=window['predicted_w'],
        mode='lines',
        name='XGBoost Predicted (W)',
        line=dict(color='#E65100', width=1.5, dash='dash'),
        hovertemplate='%{x|%Y-%m-%d %H:%M}<br>Predicted: %{y:.0f} W<extra></extra>'
    ),
    row=1, col=1
)

# Shade peak solar hours 10:00–15:00 each day with light yellow vrects
for day_offset in range(14):
    day = start_date + pd.Timedelta(days=day_offset)
    peak_start = pd.Timestamp(day.date()) + pd.Timedelta(hours=10)
    peak_end   = pd.Timestamp(day.date()) + pd.Timedelta(hours=15)
    fig.add_vrect(
        x0=peak_start, x1=peak_end,
        fillcolor='rgba(255, 253, 180, 0.35)',
        layer='below', line_width=0,
        row=1, col=1
    )

# ─────────────────────────────────────────────────────────────────────────────
# ROW 2 — Residual bar chart (green positive, red negative)
# ─────────────────────────────────────────────────────────────────────────────
residual_colors = [
    '#2E7D32' if r >= 0 else '#C62828'
    for r in window['residual_w']
]

fig.add_trace(
    go.Bar(
        x=window['datetime'],
        y=window['residual_w'],
        name='Residual (W)',
        marker_color=residual_colors,
        marker_line_width=0,
        hovertemplate='%{x|%Y-%m-%d %H:%M}<br>Error: %{y:.0f} W<extra></extra>'
    ),
    row=2, col=1
)

# Zero line on residual panel
fig.add_hline(y=0, line_color='black', line_width=1, row=2, col=1)

# ─────────────────────────────────────────────────────────────────────────────
# LAYOUT — range selector, rangeslider, titles, theme
# ─────────────────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='SolarWatch Pro — 14-Day Forecast vs Actual<br>'
             '<sup>3.57 kW Rooftop System, Karnal, Haryana</sup>',
        x=0.5, xanchor='center',
        font=dict(size=16)
    ),
    height=700,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=[
                dict(count=1,  label='1d', step='day',  stepmode='backward'),
                dict(count=3,  label='3d', step='day',  stepmode='backward'),
                dict(count=7,  label='1w', step='day',  stepmode='backward'),
                dict(step='all', label='all')
            ],
            bgcolor='#ECEFF1',
            font=dict(size=11)
        ),
        type='date'
    ),
    xaxis2=dict(
        rangeslider=dict(visible=True, thickness=0.06),
        type='date'
    )
)

fig.update_yaxes(title_text='Power (W)', row=1, col=1)
fig.update_yaxes(title_text='Error (W)', row=2, col=1)

fig.show()
fig.write_html('solar_forecast.html')
print('Saved: solar_forecast.html')

## About This Forecast

The XGBoost model was trained on **historical patterns** — specifically the relationship between  
irradiance, temperature, time-of-day, and actual AC output across 13 months of data.

**What the model does well:**
- Captures the diurnal curve shape (rise, peak, fall)
- Accounts for seasonal shifts in peak output hour
- Adjusts for temperature-induced efficiency loss

**What affects accuracy:**
- **Cloud transients** — sudden passing clouds cause sharp dips in actual power that the model cannot predict from time/temperature features alone. Adding a real-time sky-camera feed or short-interval weather forecasts would reduce this error.
- **Dust accumulation** — gradual soiling degrades the panel's actual yield below model prediction. A periodic cleaning schedule helps keep predictions aligned.
- **Model drift** — as panels age and degrade (~0.5 %/year), the model will slightly overpredict. Re-training once per season keeps it calibrated.

**The model becomes more accurate as more data accumulates.**  
Each additional month of real-world readings strengthens the seasonal coverage and helps the model better generalise to unusual weather events it has not seen before.

**Light-yellow bands** on the chart mark **peak solar hours (10:00–15:00)**, when the system  
delivers the highest output and when prediction accuracy matters most for grid balancing or battery scheduling.